## 1. Configuração de credenciais
Configura o token de autenticação do Hugging Face a partir dos secrets do Google Colab e definindo-o como variável de ambiente. Esse token é necessário para baixar modelos e datasets que exigem autenticação no Hugging Face.

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

## 2. Montagem do Google Drive
Monta o Google Drive no ambiente do Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Criação do ambiente virtual e instalação de dependências
Instala o gerenciador de pacotes `uv` e cria um ambiente virtual isolado (`/content/treino_env`) com Python 3.11. Essa decisão foi tomada devido a versão do python que roda no colabs ser incompativel com o training-hub, logo, precisamos executa-lo isoladamente. Em seguida, instala nesse ambiente as bibliotecas necessárias para o fine-tuning: `torch`/`torchvision` (compilados para CUDA 12.8), `training-hub` (com suporte a LoRA), `causal-conv1d` e `flash-linear-attention`, otimizadas para treinamento eficiente em GPU.

In [ ]:
!pip install uv
!uv venv /content/treino_env --python 3.11

!uv pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128 --python /content/treino_env
!uv pip install "training-hub[lora]" --python /content/treino_env
!uv pip install causal-conv1d --no-build-isolation --python /content/treino_env
!uv pip install flash-linear-attention --python /content/treino_env

## 4. Script de treinamento (LoRA fine-tuning)
Cria o script para o fine-tuning `meu_treinamento.py`, que será executado posteriormente no ambiente virtual isolado. Ao final, ou em caso de erro, compacta a pasta de checkpoints em um arquivo `.zip` e salva no Google Drive, garantindo que o progresso do treinamento não seja perdido.

In [ ]:
%%writefile meu_treinamento.py

import json
import os
import unsloth
import shutil
from unsloth import FastModel, FastLanguageModel
from datasets import load_dataset
from training_hub import lora_sft

NORMALIZED_DATASET = "/content/dataset_full.jsonl"
datasettreino = load_dataset("HuggingFaceH4/CodeAlpaca_20K", split="train")

with open(NORMALIZED_DATASET, "w", encoding="utf-8") as f:
    for row in datasettreino:
        record = {
            "messages": [
                {"role": "user", "content": row["prompt"]},
                {"role": "assistant", "content": row["completion"]},
            ]
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

CHECKPOINT_DIR = "/content/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

try:
  result = lora_sft(
      model_path="Qwen/Qwen3.5-4B",
      data_path=NORMALIZED_DATASET,
      ckpt_output_dir=CHECKPOINT_DIR,
      num_epochs=3,
      learning_rate=1e-4,
      lora_r=32,
      lora_alpha=64,
      max_seq_len=512,
      micro_batch_size=32,
      bf16=True,
      fp16=False,
  )

  print("Treinamento concluído.")
finally:
  shutil.make_archive(
      "/content/drive/MyDrive/qwen3.5-4B_code",
      "zip",
      "/content/checkpoints"
  )

## 5. Execução do script de treinamento
Executa o script `meu_treinamento.py` utilizando o interpretador Python do ambiente virtual criado, iniciando o fine-tuning do modelo.

In [ ]:
!/content/treino_env/bin/python meu_treinamento.py

## 6. Encerramento do ambiente de execução
Libera o runtime do Google Colab ao final da execução, desconectando a instância para economizar recursos computacionais.

In [ ]:
from google.colab import runtime
runtime.unassign()